In [ ]:
from pathlib import Path
import urllib.parse as up
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

BASE_DIR = Path("..").resolve()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models" / "bert"

device = "cuda" if torch.cuda.is_available() else "cpu"
device


In [ ]:
test_df = pd.read_csv(PROCESSED_DIR / "urls_test.csv")
phish_df = test_df[test_df["label"] == 1].reset_index(drop=True)

# Stichprobe für Performance (optional)
N = 2000
phish_sample = phish_df.sample(n=N, random_state=42).reset_index(drop=True)

len(phish_sample), phish_sample.head()


In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_DIR)
model = DistilBertForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()


In [ ]:
class URLDataset(Dataset):
    def __init__(self, urls, tokenizer, max_len=64):
        self.urls = [str(u) for u in urls]
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.urls)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.urls[idx],
            add_special_tokens=True,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze()
        }


def bert_predict(urls):
    ds = URLDataset(urls, tokenizer)
    dl = DataLoader(ds, batch_size=64)

    preds, probs = [], []

    with torch.no_grad():
        for batch in tqdm(dl, desc="BERT predict"):
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            out = model(ids, attention_mask=mask)
            logits = out.logits
            prob = torch.softmax(logits, dim=1)[:,1]
            preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
            probs.extend(prob.cpu().tolist())

    return preds, probs



In [ ]:
# Features laden
train_feat = pd.read_csv(PROCESSED_DIR / "urls_train_features.csv")
test_feat = pd.read_csv(PROCESSED_DIR / "urls_test_features.csv")

# BERT-Features aus Dateien laden
bert_train = pd.read_csv(BASE_DIR / "results" / "errors" / "../metrics" / "../.." )  # already done in training


In [ ]:
from lightgbm import Booster
hybrid_model = lgb


In [ ]:
def ensure_http(url):
    if url.startswith(("http://", "https://")):
        return url
    return "http://" + url

def attack_subdomain(url):
    u = ensure_http(url)
    p = up.urlsplit(u)
    h = "sec-check." + p.netloc
    return up.urlunsplit((p.scheme, h, p.path, p.query, p.fragment))

def attack_tracking(url):
    u = ensure_http(url)
    p = up.urlsplit(u)
    extra = "utm_source=mail&utm_campaign=security"
    new_q = extra if not p.query else p.query + "&" + extra
    return up.urlunsplit((p.scheme, p.netloc, p.path, new_q, p.fragment))

def attack_encode(url):
    u = ensure_http(url)
    p = up.urlsplit(u)
    enc_path = up.quote(p.path, safe="/")
    return up.urlunsplit((p.scheme, p.netloc, enc_path, p.query, p.fragment))

def attack_homoglyph(url):
    u = url.replace("login","Iogin").replace("paypal","paypaI")
    return u

attack_funcs = {
    "subdomain": attack_subdomain,
    "tracking": attack_tracking,
    "encode": attack_encode,
    "homoglyph": attack_homoglyph
}


In [ ]:
rows = []
for idx, row in phish_sample.iterrows():
    for atk, fn in attack_funcs.items():
        rows.append({
            "url_orig": row["url"],
            "url_adv": fn(row["url"]),
            "attack": atk
        })

adv_df = pd.DataFrame(rows)
adv_df.head()


In [ ]:
adv_preds, adv_probs = bert_predict(adv_df["url_adv"])
adv_df["bert_pred"] = adv_preds
adv_df["bert_proba"] = adv_probs


In [ ]:
# Hybrid-Feature-Feeder (vereinfachte Version)
feat_test = test_feat.set_index("url")

hyb_preds = []
hyb_probs = []

for url in adv_df["url_adv"]:
    if url in feat_test.index:
        x = feat_test.loc[url].drop(["label"])
        prob = hybrid_model.predict_proba([x])[0][1]
        pred = int(prob > 0.5)
    else:
        prob = 0.0
        pred = 0
    hyb_preds.append(pred)
    hyb_probs.append(prob)

adv_df["hyb_pred"] = hyb_preds
adv_df["hyb_proba"] = hyb_probs


In [ ]:
def attack_success(df, model):
    out = []
    for atk in attack_funcs.keys():
        sub = df[df["attack"] == atk]
        total = len(sub)
        missed = (sub[f"{model}_pred"] == 0).sum()
        out.append({
            "attack": atk,
            "total": total,
            "missed": missed,
            "success_rate": missed / total
        })
    return pd.DataFrame(out)

bert_adv = attack_success(adv_df, "bert")
hyb_adv  = attack_success(adv_df, "hyb")

bert_adv, hyb_adv


In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=bert_adv, x="attack", y="success_rate")
plt.title("BERT Adversarial Success Rate")
plt.show()

plt.figure(figsize=(10,5))
sns.barplot(data=hyb_adv, x="attack", y="success_rate")
plt.title("Hybrid Adversarial Success Rate")
plt.show()


In [ ]:
examples = adv_df[(adv_df["bert_pred"] == 0)].groupby("attack").head(3)
examples[["attack","url_orig","url_adv","bert_proba"]]
